# Neural Networks - Loss Functions

## 1. Why We Need a Loss Function

- How to know a prediction is correct?
- Minimise loss, not get the perfect prediction.
- **Loss landscape** = (weights, loss) -> gradient descent. -> update param
  - weights = where your are
  - loss = altitude
  - gradient = which direction to go downhill
  - optimiser = how to warlk
- Not only tell a prediction is wrong, but also how wrong it is.

```math
\min_{\theta} L(f(x; \theta), y)
```

## 2. Regression Losses

A loss function has 4 properties:
1. error = 0 -> loss = 0
2. error increases -> loss increase
3. loss is never negative
4. loss must be smooth (for gradient descent)

- **MAE**: gradient is 1, -1 -> always sharp like an F1 car with FULL ACCELERATION. While **MSE**'s gradient is $\frac{\partial L}{\partial \^{y}} = 2(\^{y} - y)$, which is gentler (more stable) near the optimum.
- **MSE** penalises error more than **MAE**, but can't be used for very large outliers.
- **MSE** has parabolic shape, **MAE** has sharp corner.
- **Huber Loss** is the best of both, smooth at the centre, then linear edges. For an optimum range $[-\delta, \delta]$, $e = y - \^{y}$

```math
L_{\delta}(e) = \begin{cases}
    \frac{1}{2}e^2, & |e| \le \delta \\
    \delta(|e| - \frac{1}{2}\delta), & |e| > \delta
\end{cases}
```

- **Log-Cosh Loss** also behaves like MSE for small errors, has similar shape to Huber but smoother (I guess?), and MAE for large errors. But unlike Huber, it is smooth everywhere.

```math
L = \log(\cosh(e))
```

| Property                       | MAE               | MSE                 | Huber                       | Log-Cosh                 |
| ------------------------------ | ----------------- | ------------------- | --------------------------- | ------------------------ |
| Smooth                         | ❌                 | ✅                   | Mostly                      | ✅                        |
| Robust to outliers             | ✅                 | ❌                   | ✅                           | ✅                        |
| Large errors punished strongly | ❌                 | ✅                   | Moderate                    | Moderate                 |
| Gradient continuous            | ❌                 | ✅                   | ✅                           | ✅                        |
| Common use                     | Robust regression | Standard regression | Object detection, robust ML | Some deep learning tasks |


In [1]:
import numpy as np

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def huber(y_true, y_pred, delta = 1.0):
    error = y_true - y_pred
    abs_error = np.abs(error)
    quadratic = .5 * error**2
    linear = delta * (abs_error - .5*delta)
    return np.mean(np.where(abs_error <= delta, quadratic, linear))

def log_cosh(y_true, y_pred):
    error = y_true - y_pred
    return np.mean(np.log(np.cosh(error)))

## 3. Probabilistic View of Regression Losses

> Loss functions are not invented. They are discovered from Probability Theory.

- Noise = randomness -> Add random noise into the equation:

```math
y = f(x) + \epsilon
```

- A prediction = probability -> **likelihood**. -> **Maximum Likelihood Estimation (MLE)**.

```math
\theta^* = \arg\max_{\theta} P(\text{data} | \theta)
```

- While Loss requires `min`, likelihood requires `max`.

### Gaussian

> **Assumption**: noise follows Normal Distribution, *large errors are rare, most likely error is zero*.  -> Gaussian probability density

```math
p(y|\^{y}) = \frac{1}{\sqrt{2\pi \sigma^2}}\exp(-\frac{(y - \^{y})^2}{2\sigma^2})
```

- Use $\log(ab) = \log(a) + \log(b)$ to avoid likelihood $P = \prod_i p_i$ to shrink down to zero.
- **Negative Log-Likelihood (NLL)**: optimisers minimise, not maximise -> turn log to negative.
- Substitude Gaussian and remove constants don't depend on the model params -> **Sum of Squared Error (SSE)**.

```math
\log(P) = \sum_i(y_i - \^{y}_i)^2
```

### Laplace

> **Assumption**: noise follows Laplace distribution (sharper peak, heavier tails). *Large errors happen more ofter, most likely error is zero*. -> **Laplace probability density**

```math
p(y|\^{y}) = \frac{1}{2b}\exp(-\frac{|y - \^{y}|}{b})
```

- Take the log again -> **MAE**.

```math
\log(P) = \sum_i |y_i - \^{y}_i|
```

### Bayesian

- Instead of find the best set of params -> estimate **distribution over params**.

> Choose probabilistic model of the world -> Loss follows naturally.

- Papers always start from likelihood -> loss. They ask: "What probabilistic process generated this data?"
  - Gaussian -> MSE
  - Bernoulli -> Binary Cross Entropy
  - Categorical -> Softmax Cross Entropy
  - Poisson -> Poisson loss
  - Negative Binomial -> Count-model losses
- Some hyprid distributions:
  - Mixture distributions (combining multiple probability distributions)
  - Mixture Density Networks (Bishop, 1994)
  - Custom likelihood models
  - Piecewise probability models
  - Robust Bayesian models

## 4. Binary Classification Loss

- Classification -> Probability.
- NNs train through **Gradient**, not Loss.

e.g., Activaiton = Sigmoid. output = .9 -> loss = .1 -> confident. But if output = .1 -> loss = .9 -> *Still confident* but in the opposite class.

Now if we use derivative:

```math
\frac{\partial L}{\partial z} = 2(y - \^{y})\^{y}(1 - \^{y})
```

so if $\^{y}$ reaches 0 or 1 -> derivative reaches 0 (*Gradient saturation*) -> no update.

### Likelihood

- $y = 1 \Rightarrow P(y) = \^{y}$, $y = 0 \Rightarrow P(y) = 1 - \^{y}$. So:

```math
P(y) = \^{y}^y(1 - \^{y})^{1-y}
```

### Binary Cross Entropy

- We want to maximise likelihood = minimise -(likelihood).
- Sum of logs of probability

```math
L = -\log(P) = -[y\log(\^{y}) + (1-y)\log(1 - \^{y})]
```

In [2]:
import numpy as np

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-12 # avoid log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps)

    loss = -(y_true*np.log(y_pred) + (1 - y_true)*np.log(1 - y_pred))
    return np.mean(loss)

In [3]:
y_true = np.array([1, 0, 1, 1])

y_pred = np.array([
    0.9,
    0.2,
    0.8,
    0.4
])

print(binary_cross_entropy(y_true, y_pred))

0.3669845875401002


### What makes BCE trains much faster

e.g., activation = Sigmoid

```math
\frac{\partial L}{\partial z} = \^{y} - y
```

- BCE's derivative contains only error, not sigmoid.

## 5. Multi-class Classification

- For exclusive events: $\sum_i P_i = 1$ -> Can't use Sigmoid.
  - Sigmoid treats each class indepently -> probability sum > 1
- In multi-class, outputs are called **logits**, not probabilities.
- Sum of logits = 1

### Softmax

- Properties:
  - Always positive
  - Smooth
  - Differentiable
  - Large score = large probability
- For logits $z_i$, convert them to $e^{z_i}$, then normalise:

```math
P_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
```

- Exponential = higher score, higher confidence.
- **Probability simplex**: where every probability vector is in.

In [4]:
import numpy as np

def softmax(logits):
    exp = np.exp(logits)
    return exp / np.sum(exp)

### One-hot labels

- **One-hot encoding**: turn classes into an array of 0, 1.

### Categorical Cross Entropy

```math
L = -\sum_i y_i \log p_i
```

where $p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$

e.g., y = [0, 1, 0], prediction: Dog = 0.2, cat = 0.6, horse = 0.2. Cross entropy is: $0*\log(0.2) + 1*\log(0.6) + 0*\log(0.2)$. Everything vanishes except $1*\log(0.6)$.

In [5]:
import numpy as np

def cross_entropy(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)

    return -np.sum(y_true * np.log(y_pred))

In [6]:
y_true = np.array([0, 1, 0])
y_pred = np.array([0.1, 0.8, 0.1])
loss = cross_entropy(y_true, y_pred)

print(loss)

0.2231435513142097


## 6. Why Cross Entropy Works So Well

- **BCE**'s derivative doesn't have the sigmoid part, while **MSE**'s one saturates to zero.
- Probability -> correctness + confidence.
- **Surprisal**: probability of an impossible event to occur, also $-\log(p)$
- Low probability -> High surprise -> Huge information gain.

> **Entropy**: Average amount of surprise before observing the outcome.

```math
H(P) = -\sum_i P_i \log P_i
```

- Uncertain = High entropy.

> **Cross-entropy** measures how surprised is the model by reality. Training tries to minimise this surprise.

- If a model is *overconfident* -> **calibrate** it.

### KL Divergence

- **Distributional distance** = distance between two Probability distributions. -> **KL Divergence**:

```math
D_{KL}(P||Q) = \sum_i P_i \log \frac{P_i}{Q_i}
```

> **KL Divergence**: How much extra information do I need because I used distribution $Q$ when the true distribution is $P$.

```math
H(P, Q) = H(P) + D_{KL}(P||Q)
```

- $H(P, Q)$: cross-entropy between P and Q.
- $H(P)$: entropy of the true distribution, is constant.
- $D_{KL}(P||Q)$: KL divergence.

> Minimising Cross entropy = Minimising KL Divergence.

## 7. Information Theory Behind Deep Learning

> Training = Learning the probability distribution that generated the data.

> Information depends on **surprise**.

- $P(\text{event}) = 1$ -> the event is perfectly expected -> No information.
- $P(\text{event}) = 0.0000001$ -> NO WAY the event is gonna happen -> Huge information.

### Surprisal

**Surprisal (Self-information)**

```math
I(p)
```

- Properties:
  - Rare event -> More information
  - Indepenent events add information -> $I(x) = - \log P(x)$

e.g., P = 1 -> I = 0 bits, P = 0.5 -> I = 1 bit, P = 0.25 -> I = 2 bits, P = 0.125 -> I = 3 bits.

- $P(A,B) = P(A)P(B)$ but what we want is add (information) $I(A, B) = I(A) + I(B)$. That's why log comes in: $\log(ab) = \log a + \log b$.

e.g., lottery has prob 1 in 10 million -> $-\log_2(10^{-7}) \approx 23.3$ -> Winning tells you 23 bits of information.

### Entropy

> **Assumption**: Don't know the outcome, we have distribution $P(x)$ which tells how much information we expect on average.

```math
H(P) = \sum P(x)(-\log P(x))
```

e.g., coin flip: heads = 99%, tails = 1% -> predictable -> low entropy; heads = tails = 50% -> unpredictable -> high entropy.

### Cross Entropy

> **Cross Entropy**: Total cost to encode reality using my model.

```math
H(P,Q) = -\sum P(x)\log Q(x)
```

- $Q = P$ -> perfect prediction -> cross-entropy = entropy
- Wrong prediction -> cross-entropy increases.

### KL Divergence

```math
D_{KL}(P||Q) = H(P,Q) - H(P) = \sum P(x) \log \frac{P(x)}{Q(x)}
```

- KL divergence is always >= 0. Equality at $P(x) = Q(x)$.
- KL divergence is not distance: $D_{KL}(P||Q) \ne D_{KL}(Q||P)$.
- It measures extra information required because one's beliefs are wrong.

> We don't minimise KL Divergence, we minimise Cross Entropy. Because the $H(P)$ is a fixed constant.

### Coding theory

- Compression machine.
- Common words = short codes. Rare words = long codes.
- Optimal code length: $-\log P(x)$.

### GPT = Compression machine

- Learning probabilities that minimise average code length.
- Every token contributes: $-\log P(token)$.
- Rare words (tokens) = low probability = long code length.

### Why Language Models report Perplexity

> **Perplexity**: Average number of equally likely choices remaining.

```math
2^H
```

- Perplexity = 2 -> roughly 2 possible next tokens
- Perplexity = 100 -> uncertain
- High perplexity -> Worse language model
- Low perplexity -> Better language model


### Summary

> Imagine shortest driving path from London > Edinburgh is 650km
> - **Entropy** = 650km is unavoidable distance, no matter how you travel
> - But you actually drive 730km -> This is **Cross Entropy**.
> - The difference between entropy and cross-entropy is **KL Divergence** = 730 - 650 = 80.

```
Reality
        │
        ▼
Probability Distribution P
        │
        ▼
Entropy
(average uncertainty)
        │
        ▼
Model predicts Q
        │
        ▼
Cross Entropy
        │
        ▼
Cross Entropy
=
Entropy
+
KL Divergence
        │
        ▼
Training
=
Minimise KL
        │
        ▼
Model distribution
≈
Reality
        │
        ▼
Better prediction
        │
        ▼
Better compression
```

## 8. Numerical Stability

- **Floating-point arithmetic** fails.
- Computer approximation -> underflow, overflow.

### Stable Softmax

- $e^1000$ -> inf, $e^{-1000}$ -> 0.
- Choose constant c, minus c from all logits.

```math
f(x_i) = \frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{x_i}}{\sum_j e^{x_j}}
```

both numerator and denominator cancel the factor $e^c$.

- Choose $c = \max(x_i)$.

In [1]:
import numpy as np

def softmax(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

### Stable Cross Entropy

- $p = 0$ -> loss = $\log(0)$ -> Nan
- **Clamp**/**Clip** the p: Create $\epsilon = 1e-12$
  - If $p < \epsilon$: increase p to exactly $\epsilon$.
  - If $p > 1 - \epsilon$: decrease p to exactly $1 - \epsilon$.

In [2]:
import numpy as np

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)

    return -np.mean(y_true*np.log(y_pred) + (1 - y_true)*np.log(1 - y_pred))

### The LogSumExp Trick

```math
\log(\sum_i e^{x_i})
```

- Let $m = \max(x)$ -> substract m from $x_i$.
- $x_i - m \le 0$ -> $e^{x_i - m} \le 1$.

```math
\log(\sum_i e^{x_i}) = m + \log(\sum_i e^{x_i - m})
```

> In PyTorch, `CrossEntropyLoss` requires raw logits input to avoid information loss. (Instead of computing Probability = softmax > loss = cross-entropy).

In [3]:
import numpy as np

def logsumexp(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

### LogSoftmax

- Instead of computing $\log(\text{Softmax}(x))$, we compute this:

```math
f(x_i) = x_i - \log(\sum_j e^{x_j}) = x_i - m - \log(\sum_j e^{x_j - m})
```

for m := max of x.

In [4]:
import numpy as np

def log_softmax(x):
    x = np.asarray(x, dtype=np.float64)
    return x - logsumexp(x)

### Summary

```
Large positive numbers
        │
        ▼
Exponentials overflow
        │
        ▼
Subtract maximum
        │
        ▼
Stable Softmax
        │
        ▼
Need logarithms
        │
        ▼
Use LogSumExp
        │
        ▼
Stable LogSoftmax
        │
        ▼
Stable Cross Entropy
```

## 9. Automatic Differentiation Through Losses

### MSE Gradient

```math
L = \frac{1}{2}(\^{y} - y)^2
```

Gradient:

```math
\frac{dL}{d \^{y}} = \^{y} - y
```

### MAE Gradient

```math
L = |\^{y} - y|
```

Gradient:

```math
\frac{dL}{d \^{y}} = \begin{cases}
    1, & \^{y} \ge y \\
    -1, & \^{y} < y
\end{cases}
```

- MAE gradient doesn't care about magnitude, only *direction*.

### Huber Gradient

```math
L = \begin{cases}
    \frac{1}{2}e^2, & |e| \le \delta \\
    \delta(|e| - \frac{1}{2}\delta), & |e| > \delta
\end{cases}
```

Gradient:

```math
\frac{dL}{d \^{y}} = \begin{cases}
    e, & |e| < \delta \\
    \delta\text{sign}(e), & |e| > \delta
\end{cases}
```

- Behaves like MSE near optimum, else MAE.

### Binary Cross Entropy

```math
L = -y\log(\^{y}) - (1 - y)\log(1 - \^{y})
```

Gradient:

```math
\frac{dL}{d \^{y}} = -\frac{y}{\^{y}} + \frac{1 - y}{1 - \^{y}}
```

### Cross Entropy + Softmax

With logit z:

```math
p = \text{softmax}(z)
```

The loss is:

```math
L = -\sum_i y_i \log p_i
```

Gradient: applying chain rule

```math
\frac{\partial L}{\partial z} = p - y
```

### Example

- Truth = Dog
- Model prob.: [Cat, Dog, Bird] = [0.7, 0.2, 0.1]
- One-hot target: [0, 1, 0]
- Gradients: [0.7, -0.8, 0.1]

**Interpretation**
- Reduce Cat logit
- Increase Dog logit
- Reduce Bird logit

### Comparing Gradient Behaviour

- **MSE** (after Sigmoid): sigmoid in [0, 1] -> tiny gradient -> *Slow learning*.
- **BCE**: even if confidently predicting a wrong class, gradient remains large.

### Implementation

In [5]:
import numpy as np

def mse_grad(y_true, y_pred):
    return y_pred - y_true

def mae_grad(y_true, y_pred):
    grad = np.sign(y_pred - y_true)         # direction of the gradient
    grad[np.isclose(y_pred, y_true)] = 0.0  # handle floating-point precision issues
    return grad

def huber_grad(y_true, y_pred, delta=1.0):
    error = y_pred - y_true
    grad = np.where(
        np.abs(error) <= delta,
        error,
        delta * np.sign(error)
    )

def cross_entropy_grad(probs, one_hot_labels):
    return probs - one_hot_labels

## 10. Loss Landscapes

> **Loss landscape**: The shape of the loss function over all possible parameter values.

- Every point in $\theta$ is a point, each has a height = loss.

> **Training**: Moving through loss landscape towards lower ground.

### Gradient descent = walking downhill

- $\nabla_{\theta}L(\theta)$ = which direction makes the loss rise fastest.
- $\theta \leftarrow \theta - \eta \nabla_{\theta}L(\theta)$ = negative gredient points downhill.

> The gradient is only a **local measurement**, not the full shape of the landscape. Like standing in fog on a mountain, you only feel the slopes down beneath your feet.

### Critial Points

> **Critical point**: A point where the gradient is zero. But still, it's local optimum.

$$
\nabla_{\theta}L(\theta) = 0
$$

- **Local minimum** = Nearby low point -> Upwards in every nearby direction.
- **Local maximum** = Nearby high point -> Downwards in every nearby direction.
- **Saddle point** = Neither a minimum nor maximum -> Up in some directions, down in others.

> In high-dim neural networks, saddle points are very common.

### Why local minima are ok

- Over-parameterised networks -> flexible.
- A low-loss path connect separate solutions.
- Mini-batch gradients -> SGD jiggles around -> Escape shallow traps and saddle regions.

> **Stochastic Gradient Descent (SGD)**: Update params using a single, randomly selected training example (or a small batch) at a time.

### Sharp vs Flat Minima

- **Sharp minima** (_/): Small change in params -> Huge change in loss.
- **Flat minima** (U): Small change in params -> Barely change in loss.

> In most occasions, if small changes don't affect loss much -> Flatter solutions generalise better.

### Curvature and Hessian

- Gradient = slope, seconde derivative = curvature.
  - Positive: bowl like
  - Negative: hill like
  - Near zero: nearly flat
- For one param:

$$
\frac{\partial^2 L}{\partial w^2}
$$

- **Hessian**: For many prams, Hessian's eigenvalues describe curvature.
  - All positive: local minimum
  - All negative: local maximum
  - Mixture positive + negative: saddle point
  - Very large positive: sharp direction
  - Near zero: flat direction

$$
H_{ij} = \frac{\partial^2 L}{\partial \theta_i \partial \theta_j}
$$

> In a large network, computing a full Hessian is enormous -> Hessian is used to *explain optimisation behaviour in geometry*.

### Ill-conditioning: why optimisation can zig-zag

- Look at contours from above.
- Long thin valley, one side has high curvature, the other is flat. Gradient step bounces from side to side.
- **Ill-conditioning**: Due to high learning rate, it steps from wall to wall.
- That's why we need:
  - learning rate modification
  - feature normalisation
  - batch normalisation
  - momentum
  - Adam-like optimisers

### Why loss landscapes have symmetry

- Multiple param settings to represent the same function.
- A solution is not an isolated point, but a whole family of equivalent points or directions in param space. - > *Landscapes have many flat directions*.
- Thought experiment:

For,

$$
\^{y} = wx + b
$$

with MSE loss, the landscape $(w,b)$ is convex -> one global best region. But now we add more hidden layers, non-linear activations -> Non-convex loss -> Lanscape contains saddle points, flat regions, many equivalent low-loss solutions. So:
- Linear regression: Params affect output in simple convex way
- Deep network: layers multiply + non-lineariry -> complex geometry.

## 11. Modern Loss Functions

Modern loss functions ask:

> What behaviour do we want to model to learn?

### Focal loss - focus on hard examples

- For a correct example is tiny, using Binary cross-entropy:

$$
\text{BCE} = -\log(p_t)
$$

where $p_t$ is probability of the true class. If $p_t = .99$ then $\text{BCE} \approx 0.010$.

- Now for a highly imbalanced dataset (e.g., 99% background pixels—millions of easy background examples can still dominate total training), Focal Loss deliberately reduces their contribution

$$
\text{FL}(p_i) = -(1 - p_t)^{\gamma}\log(p_t)
$$

where:
- $\gamma \ge 0$: focusing param
  - if $\gamma = 0$: this is ordinary BCE
- if $p_t$ is high: $(1 - p_t)^{\gamma}$ is tiny
- if $p_t$ is low, the multiplier remains large.

> **Focal Loss** is useful when most candiate regions are easy negative.

We can even add **class weighting** $\alpha_t$ if a class is intrinsically rare or important:

$$
\text{FL}(p_t) = -\alpha(1 - p_t)^{\gamma}\log(p_t)
$$

### Label smoothing - do not demand impossible certainty

- One-hot labels are not perfectly 0 and 1 (e.g., blurry images)
- **Label smoothing** replaces the one-hot target with softer distribution.

$$
y_k' = (1 - \epsilon)y_k + \frac{\epsilon}{K}
$$

where:
- $\epsilon$: small smoothing value (e.g., 0.1)
- $K$: number of classes.
- For labels $[0, 0, 1, 0]$ -> $[0.25, 0.25, 0.95, 0.25]$: the class 3 is highly correct but the others are also possible.

### Dice loss - optimise overlap, not isolated pixels

- Medical image segmentation, 1% pixels is tumour.
- Model predicts "background everywhere" and achieves 99% accuracy -> Useless!!!
- **Dice Coefficient** measures region overlap:

$$
\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}
$$

- For soft NN predictions $p_i$ and labels $y_i$:

$$
\text{Dice} = \frac{2 \sum_i p_iy_i + \epsilon}{\sum_i p_i + \sum_i y_i + \epsilon}
$$

then:

$$
\text{DiceLoss} = 1 - \text{Dice}
$$

> **Dice Loss** directly rewards predicted possible area overlapping real positive area. Unlike BCE sees the prediction as a whole mask.

### IoU Loss - stricter region overlap

- Intersection over Union (IoU), also called as Jacobian Index

$$
\text{IoU} = \frac{|A \cap B|}{|A \cup B|}
$$

Since $|A \cup B| = |A| + |B| - |A \cap B|$, the soft form is:

$$
\text{IoU} = \frac{\sum_i p_i y_i + \epsilon}{\sum_ip_i + \sum_iy_y - \sum_ip_iy_i + \epsilon}
$$

and

$$
\text{IoULoss} = 1 - \text{IoU}
$$

### Metric-learning losses - learn a useful geometry

- Input mapped to embedding
- Similar examples near each other, different ones are apart.

#### Contrastive loss

A pair $(x_1, x_2)$, distance:

$$
D = ||f(x_1) - f(x_2)||_2
$$

- Similar pairs -> minimise D.
- Dissimilar pairs -> margin m:

$$
L = yD^2 + (1 - y)\max(0, m - D)^2
$$

- $y = 1$ -> similar
- $y = 0$ -> different

#### Triplet loss

- A triplet contains:
  - a := anchor
  - p > 0 := same identity or meaning
  - n < 0 := different identity or meaning

$$
d(a, p) + m < d(a, n)
$$

Loss:

$$
L = \max(0, d(a, p) - d(a, n) + m)
$$

> The positive must be closer to the anchor than the negative by margin m.

### Cosine loss - care about direction, not length

Cosine similarity:

$$
\cos(\theta) = \frac{u \cdot v}{\| u \| \| v \|}
$$

Cosine loss:

$$
L = 1 - \cos(\theta)
$$

### InfoNCE - the engine behind contrastive representation learning

> **InfoNCE**: Assign the greatest similarity to the true match among all candidate matches.
> Cross-entropy over candidate matches.

For anchor $a$, a positive $p$, and negatives $n_1,...,n_N$:

$$
L = -\log \frac{\exp(\text{sim}(a, p)/\tau)}{\exp(\text{sim}(a, p)/\tau) + \sum_j\exp(\text{sim}(a, n_j)/\tau)}
$$

- $\text{sim}$: cosine similarity or dot product.
- $\tau$: temperature, controlling how sharply similarities are compared.

### Summary

| Loss | Behaviour it encodes |
|---|---|
| Focal Loss | Prioritise hard examples |
| Label smoothing | Avoid unjustified certainty |
| Dice / IoU | Maximise region overlap |
| Contrastive / Triplet | Organise an embedding space |
| Cosine Loss | Align semantic direction |
| InfoNCE | Identify the true match among alternatives |

## 12. Research Frontier

### Self-supervised learning: create supervision from the data

> Raw data contains structure.

- Data augmentation defines the invariances learned.
- SimCLR

### Diffusion objectives: learn to reverse corruption

- Raw $x_0$ -> add noise -> $x_t$ -> learn to predict the noise

$$
L_{\text{diffusion}} = \mathbb{E}_{x_0, \epsilon, t}[\|\epsilon - \epsilon_{\theta}(x_t, t)\|^2]
$$

- It starts from pure noise > repeatedly denoise to generate a sample.

### Energy-based models: learn what looks plausible

- Assign energy to every input:

$$
E_{\theta}(x)
$$

- Desired input -> low energy
- Implausible input -> high energy

Probability:

$$
p_{\theta}(x) = \frac{\exp(-E_{\theta}(x))}{Z_{\theta}}
$$

- $Z_{\theta}$: partition function, normalises all probabilities.

### Reinforcement learning objectives: consequences arrive later

- In reinforcement learning, an action changes the future -> maximise expected total reward:

$$
J(\pi) = \mathbb{E}_{\tau~\pi}[\sum_{t = 0}^T \gamma^tr_t]
$$

where:
- $\pi$: the policy
- $r_t$: reward at time t
- $\gamma$: discounts distant rewards
- $\tau$: possible trajectory.

### Reward models: turn preferences into a learnable signal

Suppose a human prefers response $y_w$ over $y_l$ for prompt $x$. A reward model $r_{\phi}$ can be trained with:

$$
L_{\text{reward}} = -\log \sigma(r_{\phi}(x, y_w) - r_{\phi}(x, y_l))
$$

- Binary cross-entropy applied to the difference in scores.
- Make the preferred response receive higher reward than the rejected response.

### Direct Preference Optimisation (DPO)

> **DPO** avoids explicitly training a separate reward model during this final optimisation.

Given preferred $y_w$, rejected $y_l$, reference policy $\pi_{\text{ref}}$ and current policy $\pi_{\theta}$, it optimises:

$$
L_{DPO} = -\log \sigma(\beta[
    \log\frac{\pi_{\theta}(y_w|x)}{\pi_{\text{ref}}(y_w|x)} -
    \log\frac{\pi_{\theta}(y_l|x)}{\pi_{\text{ref}}(y_l|x)}
])
$$

### Loss shaping and reward hacking

A loss is always an imperfect proxy for the true goal.

If we reward a cleaning robot only for making a room appear clean, it might hide rubbish under a carpet. The loss decreased; the real objective failed.

This is **reward hacking** or **specification gaming**.

Research therefore asks:
- Does the objective capture what humans actually value?
- Can the model exploit loopholes?
- Does a learned reward generalise outside its training comparisons?
- How should we trade off helpfulness, safety, truthfulness, and uncertainty?

The hard problem is not merely finding a differentiable number. It is choosing a number whose optimisation produces good behaviour.

### Learnable losses and meta-learning

A meta-learning formulation has two levels:

$$
\theta^*(\phi) = \arg\min_{\theta}L_{\text{train}}(\theta;\phi) \\
\min_{\phi}L_{\text{validation}}(\theta^*(\phi))
$$

for:
- $\theta$: ordinary model params
- $\phi$: params controlling the learning rule or loss
- inner loop: train the model
- outer loop: improve how it learns.

### Summary

| Objective type | The question encoded by its loss |
|---|---|
| Cross-entropy | Which label is correct? |
| InfoNCE | Which candidate truly matches this input? |
| Diffusion MSE | What noise was added to this sample? |
| Reward modelling | Which output do humans prefer? |
| DPO | Which response should become relatively more likely? |
| RL return | Which sequence of actions produces the best long-term consequences? |
| Energy objective | Which states should be plausible or desirable? |

## 13. Final project